# [학습 정리] Django ORM과 1:N 관계 심화: 저자-도서 관리 시스템
1. 프로젝트 초기 환경 설정
    웹 프로젝트를 시작할 때 가장 먼저 해야 할 일은 '격리된 환경'을 만들고 불필요한 파일이 서버에 올라가지 않도록 막는 것입니다.

    가상환경(venv) 생성 및 활성화: 프로젝트마다 요구하는 패키지 버전이 다르기 때문에, 충돌을 막기 위해 반드시 가상환경을 분리합니다.

    .gitignore 설정: DB 파일(db.sqlite3), 가상환경 폴더(venv/), 파이썬 캐시(__pycache__/) 등은 Git 저장소에 올라가면 보안 사고나 충돌을 유발하므로 초기 단계에 반드시 제외해야 합니다.

In [ ]:
# 터미널 명령어 (참고용)
python -m venv venv
source venv/Scripts/activate  # Windows 기준
pip install django
# 이후 .gitignore 파일 생성 후 환경 파일들 등록

2. 데이터베이스 모델 설계 (1:N 관계)
    어제 배운 다이어리-댓글 관계와 완벽히 동일한 구조입니다. 저자(Author) 1명이 여러 권의 책(Book) N을 집필하는 1:N 구조를 설계합니다.

    SQL 관점 vs Django ORM 관점
    관계형 DB에서는 책 테이블에 author_id라는 외래키(Foreign Key) 컬럼을 두어 저자를 식별합니다. Django에서는 이를 객체 지향적으로 참조합니다.

In [ ]:
# libraries/models.py
from django.db import models

class Author(models.Model):
    name = models.CharField(max_length=100)
    # ... 기타 필드 생략

class Book(models.Model):
    """
    [Model: Book]
    - Author 모델과 1:N 관계 설정
    - on_delete=models.CASCADE: 저자 정보가 삭제되면 그 저자의 책 데이터도 연쇄 삭제
    """
    author = models.ForeignKey(Author, on_delete=models.CASCADE)
    title = models.CharField(max_length=200)
    # ... 기타 필드 생략

3. ModelForm 설계: 사용자 입력 제어
    사용자가 책을 등록할 때, "어떤 저자의 책인지"를 직접 드롭다운으로 고르게 하는 것은 UX/UI 적으로도, 보안상으로도 치명적인 실수입니다. 이미 '특정 저자의 상세 페이지'에 들어와 있으므로, 저자 정보는 시스템이 알아서 채워 넣어야 합니다.

* **exclude 속성의 활용**

In [ ]:
# libraries/forms.py
from django import forms
from .models import Book

class BookForm(forms.ModelForm):
    class Meta:
        model = Book
        # author 필드를 화면의 입력 폼에서 제외 (시스템이 백단에서 주입할 예정)
        exclude = ('author',)

4. URL 라우팅 설계 (Variable Routing)
    "누구의 책을 만들 것인가?"에 대한 정보를 URL 자체에 담아서 서버로 보냅니다. 이를 동적 라우팅(Variable Routing)이라고 합니다.

In [ ]:
# libraries/urls.py
from django.urls import path
from . import views

app_name = 'libraries'
urlpatterns = [
    # 저자 상세 페이지
    path('<int:author_pk>/', views.author_detail, name='author_detail'),
    
    # 책 생성 처리용 URL (요구사항 반영)
    path('<int:author_pk>/books/create/', views.book_create, name='book_create'),
]

5. View 로직: 데이터 조립과 commit=False (핵심)
    가장 에러가 많이 나는 구간입니다. 화면(Form)에서는 저자(author) 필드를 빼버렸기 때문에, 들어온 데이터를 그대로 DB에 저장(save())하려고 하면 "NOT NULL constraint failed (author_id 필드가 비어있음)" 에러가 발생합니다.

    이를 해결하기 위해 20년 차 시니어들도 매번 쓰는 장고의 필살기가 바로 commit=False입니다.

In [ ]:
# libraries/views.py
from django.shortcuts import render, redirect, get_object_or_404
from .models import Author, Book
from .forms import BookForm

def book_create(request, author_pk):
    # 1. URL로 넘어온 PK 값을 통해 "어떤 저자"인지 DB에서 꺼내옴
    author = get_object_or_404(Author, pk=author_pk)
    
    if request.method == 'POST':
        form = BookForm(request.POST)
        if form.is_valid():
            # [핵심] commit=False: DB에 당장 INSERT 쿼리를 쏘지 말고, 
            # 파이썬 메모리상에 조립 중인 객체(book)만 잠깐 줘봐라!
            book = form.save(commit=False)
            
            # 비어있는 책의 저자(author) 속성에 아까 찾아둔 저자 객체를 직접 끼워 넣음
            book.author = author
            
            # 이제 빈칸이 없으므로 최종적으로 DB에 저장 (진짜 INSERT 쿼리 실행)
            book.save()
            
            # 생성 완료 후 해당 저자의 상세 페이지로 리다이렉트
            return redirect('libraries:author_detail', author.pk)

6. Template 화면 구성: 역참조(Reverse Relation)의 활용
    Author 모델에는 book이라는 필드가 없지만, 1:N 관계를 파악한 장고가 부모(Author)에게 자식(Book)들을 불러올 수 있는 book_set 이라는   리모컨(매니저)을 제공합니다. (어제 배운 JOIN 쿼리의 자동화입니다.)

    *6.1* 저자 전체 목록 (책 개수 출력)
        book_set.count를 사용하여 해당 저자를 참조하고 있는 자식 데이터의 총량을 구합니다.

In [ ]:
<h1>저자 목록</h1>
<ul>
  {% for author in authors %}
    <li>
      <a href="{% url 'libraries:author_detail' author.pk %}">
        {{ author.name }}
      </a>
      (출간 도서 수: {{ author.book_set.count }}권)
    </li>
  {% endfor %}
</ul>

*6.2* 저자 상세 페이지 (책 목록 및 폼 출력)
        book_set.all을 사용하여 자식 데이터 전체 리스트를 순회하며 출력합니다.

In [ ]:
<h1>{{ author.name }}의 상세 페이지</h1>
<hr>

<h3>새 도서 등록</h3>
<form action="{% url 'libraries:book_create' author.pk %}" method="POST">
  {% csrf_token %}
  {{ book_form.as_p }}
  <input type="submit" value="등록하기">
</form>

<hr>

<h3>집필한 도서 목록</h3>
<ul>
  {% for book in author.book_set.all %}
    <li>{{ book.title }}</li>
  {% empty %}
    <li>아직 등록된 도서가 없습니다.</li>
  {% endfor %}
</ul>